# PSXVoice2MPQ: voice super-resolution (VoiceFixer)

Runs the same VoiceFixer ONNX model as the app on a Colab GPU.
アプリと同じ VoiceFixer の ONNX モデルを Colab の GPU で実行します。

1. **Runtime → Change runtime type → T4 GPU** を選ぶ / select a T4 GPU runtime.
2. In the app, click **Export for Colab**. `sr_input.zip` is written to the output folder.
   アプリで「Colab 用にエクスポート」を押すと、出力フォルダに `sr_input.zip` ができます。
3. **Runtime → Run all**. Upload `sr_input.zip` when asked (or put it in Google Drive, see below).
   「ランタイム → すべてのセルを実行」で実行し、聞かれたら `sr_input.zip` をアップロードします。
4. `sr_output.zip` is downloaded at the end. Import it in the app with **Import Colab result**, then build.
   最後に `sr_output.zip` がダウンロードされます。アプリで「Colab の結果をインポート」してからビルドします。

With `USE_DRIVE`, the files are read from and written to `MyDrive/PSXVoice2MPQ/`, and finished voices are kept
there, so a disconnected session continues where it stopped.
`USE_DRIVE` をオンにすると Google Drive の `MyDrive/PSXVoice2MPQ/` を使います。処理済みの音声も Drive に残るので、
接続が切れても続きから再開できます。

Model: [VoiceFixer](https://github.com/haoheliu/voicefixer) by Haohe Liu et al. (code MIT, weights CC BY 4.0).

In [ ]:
# @title Settings / 設定
USE_DRIVE = False  # @param {type:"boolean"}

In [ ]:
# @title Setup: install ONNX Runtime and download the model / セットアップ
import hashlib, io, json, os, sys, time, urllib.request, wave, zipfile
import numpy as np

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ONNX Runtime 1.23 is built for cuDNN 9.8. The newer cuDNN that comes with Colab fails on
    # some convolutions (CUDNN_FE HEURISTIC_QUERY_FAILED), so install the matching version.
    # Warnings about torch's dependencies can be ignored: this notebook does not use torch.
    # ONNX Runtime 1.23 は cuDNN 9.8 向けです。Colab の新しい cuDNN では畳み込みが失敗するため、合う版を入れます。
    # torch の依存関係についての警告は無視して構いません（このノートブックは torch を使いません）。
    !pip -q install onnxruntime-gpu==1.23.2 nvidia-cudnn-cu12==9.8.0.87
import onnxruntime as ort

MODEL_ID = "voicefixer-v1"
MODEL_URL = "https://github.com/bubio/PSXVoice2MPQ/releases/download/model-data-voicefixer-1"
MODEL_FILES = {
    "vf_front.onnx": "f805f0c1b4735fd9020c4329a5c83620c23c76f34ba0ff3e9cc2320ab5ab38d3",
    "vf_back.onnx": "30789b80cc549d4d62539d745056c7046ec1c72832edccac310d67410a60e98b",
    "vf_post.onnx": "baf3a84f00751121c1ad3c2eeb7be9437ca85602763a5b0165295cc9927720e6",
}
MODEL_DIR = os.environ.get("SR_MODEL_DIR", "/content/models" if IN_COLAB else "models")

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

os.makedirs(MODEL_DIR, exist_ok=True)
for name, digest in MODEL_FILES.items():
    path = os.path.join(MODEL_DIR, name)
    if not (os.path.exists(path) and sha256_of(path) == digest):
        print("downloading", name)
        urllib.request.urlretrieve(f"{MODEL_URL}/{name}", path)
        if sha256_of(path) != digest:
            raise RuntimeError(f"SHA-256 mismatch: {name}")

if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()  # CUDA/cuDNN libraries installed with pip
# Every voice has a different length, so GPU memory is allocated in new sizes each time. Allocate
# exactly what is asked for and give it back after each run, or the T4 runs out of memory.
# 音声ごとに長さが違うため、GPU メモリを必要な分だけ確保し、実行ごとに返します（返さないと T4 のメモリが尽きます）。
providers = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in ort.get_available_providers()]
providers = [(p, {"arena_extend_strategy": "kSameAsRequested"}) if p == "CUDAExecutionProvider" else p
             for p in providers]
sessions = [ort.InferenceSession(os.path.join(MODEL_DIR, f), providers=providers)
            for f in ("vf_front.onnx", "vf_back.onnx", "vf_post.onnx")]
for s in sessions:
    # Fail instead of silently retrying on the CPU, which would take hours.
    s.disable_fallback()
active = sessions[1].get_providers()[0]
RUN_OPTIONS = ort.RunOptions()
if active == "CUDAExecutionProvider":
    RUN_OPTIONS.add_run_config_entry("memory.enable_memory_arena_shrinkage", "gpu:0")
print("ONNX Runtime", ort.__version__, "on", active)
if active == "CUDAExecutionProvider":
    from importlib.metadata import version
    print("cuDNN (pip)", version("nvidia-cudnn-cu12"))
if active != "CUDAExecutionProvider":
    print("WARNING: no GPU. Select Runtime > Change runtime type > T4 GPU. / GPU が使われていません。")

In [ ]:
# @title VoiceFixer pipeline (same as tools/voicefixer_onnx/pipeline.py) / 処理本体
DELAY, HOP, WIN, CHUNK, CTX = 1244, 441, 3072, 1536, 768

def restore(x11):
    front, back, post = sessions
    n44 = len(x11) * 4
    unet_in, orig44 = front.run(None, {"wav11k": x11[None, None, :]}, RUN_OPTIONS)
    t = unet_in.shape[2]
    if t <= WIN:
        y = back.run(None, {"unet_in": unet_in}, RUN_OPTIONS)[0][0, 0]
    else:
        outs = []
        for f0 in range(0, t, CHUNK):
            f1 = min(t, f0 + CHUNK)
            s0, s1 = max(0, f0 - CTX), min(t, f1 + CTX)
            yc = back.run(None, {"unet_in": np.ascontiguousarray(unet_in[:, :, s0:s1])}, RUN_OPTIONS)[0][0, 0]
            outs.append(yc[(f0 - s0) * HOP:] if f1 == t else yc[(f0 - s0) * HOP:(f1 - s0) * HOP])
        y = np.concatenate(outs)
    y = np.pad(y, (0, max(0, DELAY + n44 - len(y))))[DELAY:DELAY + n44]
    y = post.run(None, {"orig44k": orig44, "vf44k": y[None, None, :].astype(np.float32)}, RUN_OPTIONS)[0][0, 0]
    peak = np.abs(y).max()
    return y / peak if peak > 1.0 else y

def read_wav(data):
    with wave.open(io.BytesIO(data)) as w:
        assert w.getnchannels() == 1 and w.getsampwidth() == 2 and w.getframerate() == 11025
        return np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float32) / 32768.0

def wav_bytes(y, rate=44100):
    # Same scaling as the app (wav_io.dart).
    pcm = np.clip(np.round(y * 32768.0), -32768, 32767).astype("<i2")
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        w.writeframes(pcm.tobytes())
    return buf.getvalue()

# Check the GPU once with 35 s of noise (long enough to be split into chunks).
if active == "CUDAExecutionProvider":
    t0 = time.time()
    try:
        restore(np.random.default_rng(0).uniform(-0.1, 0.1, 11025 * 35).astype(np.float32))
    except Exception as e:
        raise RuntimeError(
            "The GPU check failed. Choose Runtime > Disconnect and delete runtime, then Run all again. "
            "If it still fails, please report this error. "
            "GPU での確認に失敗しました。「ランタイム → ランタイムを接続解除して削除」のあと、"
            "もう一度すべてのセルを実行してください。それでも失敗する場合は、このエラーを開発者に知らせてください。") from e
    print(f"GPU check OK: 35 s of audio in {time.time() - t0:.1f} s")

In [ ]:
# @title Input: upload sr_input.zip / 入力
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/PSXVoice2MPQ"
    INPUT = os.path.join(WORK, "sr_input.zip")
    if not os.path.exists(INPUT):
        raise FileNotFoundError(f"Put sr_input.zip in {WORK} / {WORK} に sr_input.zip を置いてください")
elif IN_COLAB:
    from google.colab import files
    WORK = "/content/work"
    os.makedirs(WORK, exist_ok=True)
    uploaded = files.upload()
    name = next(iter(uploaded))
    INPUT = os.path.join(WORK, "sr_input.zip")
    with open(INPUT, "wb") as f:
        f.write(uploaded[name])
else:
    INPUT = os.environ["SR_INPUT"]
    WORK = os.environ.get("SR_WORK", "work")

with zipfile.ZipFile(INPUT) as z:
    manifest = json.loads(z.read("manifest.json"))
if manifest["modelId"] != MODEL_ID:
    raise ValueError(f"This zip is for model {manifest['modelId']}, but this notebook runs {MODEL_ID}. "
                     "Open the notebook from the app that exported it.")
print(len(manifest["files"]), "files from app", manifest.get("appVersion"))

In [ ]:
# @title Process / 処理（Drive を使うと、途中で切れても続きから再開できます）
OUT_DIR = os.path.join(WORK, "out")
os.makedirs(OUT_DIR, exist_ok=True)
todo = [f for f in manifest["files"] if not os.path.exists(os.path.join(OUT_DIR, f["sha256"] + ".wav"))]
print(f"{len(manifest['files']) - len(todo)} done, {len(todo)} to go")
start = time.time()
with zipfile.ZipFile(INPUT) as z:
    for i, f in enumerate(todo, 1):
        y = restore(read_wav(z.read(f"wav/{f['sha256']}.wav")))
        target = os.path.join(OUT_DIR, f["sha256"] + ".wav")
        with open(target + ".part", "wb") as out:
            out.write(wav_bytes(y))
        os.replace(target + ".part", target)
        if i % 20 == 0 or i == len(todo):
            elapsed = time.time() - start
            print(f"{i}/{len(todo)}  {f['name']}  {elapsed / 60:.1f} min, about {elapsed / i * (len(todo) - i) / 60:.0f} min left")

In [ ]:
# @title Output: download sr_output.zip / 出力
OUTPUT = os.path.join(WORK, "sr_output.zip")
with zipfile.ZipFile(OUTPUT, "w", zipfile.ZIP_STORED) as z:
    z.writestr("manifest.json", json.dumps(manifest))
    for f in manifest["files"]:
        z.write(os.path.join(OUT_DIR, f["sha256"] + ".wav"), f"wav/{f['sha256']}.wav")
print("wrote", OUTPUT)
if IN_COLAB and not USE_DRIVE:
    files.download(OUTPUT)
elif USE_DRIVE:
    print("Download sr_output.zip from Google Drive (MyDrive/PSXVoice2MPQ). / Drive からダウンロードしてください。")